# 💊 Clinical-Grade Drug Interaction Knowledge Base — RAG Ingestion Pipeline  

<span style="color:red">by Ridwan Oladipo, MD | Medical AI Specialist</span>  

Production-grade ingestion pipeline unifying **191,541 DrugBank interactions** with **RxNorm mappings** into a clean, clinically reliable DDI knowledge base:  

- **Hierarchical normalization** (brand/generic/synonym → ingredient via RxCUI priority: IN > BN > PIN > SY)
- **Synonym unification** (1,369 acetaminophen variants → RxCUI 161)
- **Multi-tier resolution** (local cache → reverse index → RxNav API fallback)  
- **Crash-safe caching** (77,518 brand→ingredient links, 6,409 ingredient names)  
- **Atomic persistence** for multi-day API runs  
- **Bidirectional lookup architecture** (name↔RxCUI)  

🚀 Achieves **~90% pair-level mapping coverage**, yielding **170K+ verified interaction pairs** — powering RAG-based drug safety reasoning with clinical precision and production reliability.  

>⚕️ **Clinical safety meets engineering excellence** — resolving synonym chaos into a structured, high-trust pharmacologic graph.

## 📦 Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import pickle
import time
import re
from collections import defaultdict, Counter
from pathlib import Path

print("✅ Imports loaded")

✅ Imports loaded


## 📂 Load RxNorm Mappings

In [2]:
rxnorm_df = pd.read_csv('data/rxnorm_mappings.csv')

print(f"Shape: {rxnorm_df.shape}")
print(f"\nType distribution:\n{rxnorm_df['type'].value_counts()}")
print(f"\nSample:\n{rxnorm_df.head()}")

Shape: (178731, 4)

Type distribution:
type
DP            38607
SY            28324
PSN           21229
SCD           12033
SCDC          10146
SBD            8087
SU             7858
SBDC           7034
SBDG           6709
SCDG           6272
IN             5785
SCDF           5551
SBDF           5000
BN             4141
SCDGP          2480
SCDFP          1995
PIN            1895
SBDFP          1768
MTH_RXN_DP     1541
MIN             982
BPCK            696
GPCK            587
PT               11
Name: count, dtype: int64

Sample:
   rxcui type            name       name_norm
0     38   BN        Parlodel        parlodel
1     44   IN           mesna           mesna
2     44   SU           mesna           mesna
3     61   IN    beta-alanine    beta-alanine
4     61   SU  .BETA.-ALANINE  .beta.-alanine


## 🔍 Build Primary Lookup Dictionaries

In [3]:
TYPE_PRIORITY = {'IN': 1, 'BN': 2, 'SN': 3, 'PSN': 4, 'PIN': 5, 'SCD': 6, 'SBD': 7}

name_to_rxcui = {}
rxcui_to_names = defaultdict(set)

for _, row in rxnorm_df.iterrows():
    rxcui = row['rxcui']
    name_norm = row['name_norm']
    drug_type = row['type']

    rxcui_to_names[rxcui].add(name_norm)

    if name_norm in name_to_rxcui:
        existing_priority = TYPE_PRIORITY.get(name_to_rxcui[name_norm][1], 99)
        new_priority = TYPE_PRIORITY.get(drug_type, 99)
        if new_priority < existing_priority:
            name_to_rxcui[name_norm] = (rxcui, drug_type)
    else:
        name_to_rxcui[name_norm] = (rxcui, drug_type)

print(f"✅ Built lookup dicts:")
print(f"   Unique names: {len(name_to_rxcui):,}")
print(f"   Unique RxCUIs: {len(rxcui_to_names):,}")

# Test
test_drugs = ['aspirin', 'tylenol', 'acetaminophen']
print(f"\n🔍 Test lookups:")
for drug in test_drugs:
    if drug in name_to_rxcui:
        rxcui, dtype = name_to_rxcui[drug]
        print(f"   {drug} → RxCUI {rxcui} ({dtype})")

✅ Built lookup dicts:
   Unique names: 157,971
   Unique RxCUIs: 82,134

🔍 Test lookups:
   aspirin → RxCUI 1191 (IN)
   tylenol → RxCUI 202433 (BN)
   acetaminophen → RxCUI 161 (IN)


## 🌐 Build Brand→Ingredient Cache (RxNav API)

##### **Note:** This cell takes ~3-4 days on first run to cache 77K+ brand/synonym→ingredient mappings.
##### Subsequent runs load from cache instantly.

In [4]:
def normalize_drug(name):
    """Normalize drug name: lowercase, strip, remove extra spaces"""
    return re.sub(r'\s+', ' ', name.lower().strip())


def get_ingredient_from_api(rxcui, max_retries=3):
    """Resolve BN/SY/PIN/PSN RxCUI → IN RxCUI via RxNav API"""
    for attempt in range(max_retries):
        try:
            url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/related.json?tty=IN"
            resp = requests.get(url, timeout=10)

            if resp.status_code == 429:
                time.sleep(2)
                continue

            data = resp.json()
            if 'relatedGroup' in data:
                for group in data['relatedGroup'].get('conceptGroup', []):
                    if group.get('tty') == 'IN' and 'conceptProperties' in group:
                        ing_rxcui = group['conceptProperties'][0]['rxcui']
                        ing_name = group['conceptProperties'][0]['name']
                        return ing_rxcui, ing_name
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1)
                continue

    return None, None


def atomic_save(obj, path):
    """Atomic save to prevent corruption on crash"""
    temp_file = path.with_suffix(".tmp")
    with open(temp_file, "wb") as f:
        pickle.dump(obj, f)
    temp_file.replace(path)


# Load or build cache
cache_file = Path("data/bn_to_in_map.pkl")
if cache_file.exists():
    with open(cache_file, "rb") as f:
        bn_to_in_map = pickle.load(f)
    print(f"✅ Loaded existing cache: {len(bn_to_in_map):,} entries")
else:
    print("⏳ Building brand→ingredient cache via API (this will take ~2-3 hours)...")

    non_in_df = rxnorm_df[rxnorm_df['type'] != "IN"]
    unique_non_in = non_in_df[['rxcui', 'name_norm']].drop_duplicates(subset='rxcui')

    bn_to_in_map = {}
    total = len(unique_non_in)
    processed = 0

    for idx, row in unique_non_in.iterrows():
        rxcui = str(row['rxcui'])

        if rxcui in bn_to_in_map:
            processed += 1
            continue

        ing_rxcui, ing_name = get_ingredient_from_api(rxcui)
        processed += 1

        if ing_rxcui:
            bn_to_in_map[rxcui] = (ing_rxcui, ing_name)

        if processed % 50 == 0:
            atomic_save(bn_to_in_map, cache_file)
            print(
                f"Progress: {processed}/{total} processed, {len(bn_to_in_map):,} mapped ({100 * len(bn_to_in_map) / processed:.1f}% success)")

        time.sleep(0.2)

    atomic_save(bn_to_in_map, cache_file)
    print(f"✅ Final: {len(bn_to_in_map):,}/{total:,} entries ({100 * len(bn_to_in_map) / total:.1f}% coverage)")

✅ Loaded existing cache: 77,518 entries


## 🔄 Build Reverse Index (Ingredient Names)

In [5]:
ingredient_name_to_rxcui = {}

for brand_rxcui, (ing_rxcui, ing_name) in bn_to_in_map.items():
    norm_name = normalize_drug(ing_name)
    if norm_name not in ingredient_name_to_rxcui:
        ingredient_name_to_rxcui[norm_name] = ing_rxcui

print(f"✅ Reverse index: {len(ingredient_name_to_rxcui):,} unique ingredient names")

# Merge previous API discoveries
api_cache_file = Path("data/new_api_ingredients.pkl")
if api_cache_file.exists():
    with open(api_cache_file, "rb") as f:
        api_discoveries = pickle.load(f)
    ingredient_name_to_rxcui.update(api_discoveries)
    print(f"✅ Merged {len(api_discoveries):,} previous API discoveries")
    print(f"📊 Total ingredient cache: {len(ingredient_name_to_rxcui):,}")
else:
    print("ℹ️  No previous API discoveries found (first run)")

ingredient_name_to_rxcui['paracetamol'] = '161'
name_to_rxcui['paracetamol'] = ('161', 'IN')

✅ Reverse index: 6,026 unique ingredient names
✅ Merged 410 previous API discoveries
📊 Total ingredient cache: 6,405


## 🧪 Validate Unification

In [6]:
# Show top ingredients with most brand variants
collapse_check = Counter(bn_to_in_map.values())
print("Top 10 ingredients by brand/synonym count:")
for (ing_rxcui, ing_name), count in collapse_check.most_common(10):
    print(f"  {ing_name:40} ({ing_rxcui}) ⇢ {count} variants")

# Verify acetaminophen unification
brands_to_161 = [bn for bn, (ing, _) in bn_to_in_map.items() if ing == '161']
print(f"\n✅ Acetaminophen (RxCUI 161): {len(brands_to_161)} brand variants unified")

Top 10 ingredients by brand/synonym count:
  acetaminophen                            (161) ⇢ 1369 variants
  benzocaine                               (1399) ⇢ 550 variants
  lidocaine                                (6387) ⇢ 510 variants
  hydrocortisone                           (5492) ⇢ 470 variants
  diphenhydramine                          (3498) ⇢ 437 variants
  calcium carbonate                        (1897) ⇢ 436 variants
  menthol                                  (6750) ⇢ 436 variants
  aspirin                                  (1191) ⇢ 431 variants
  vitamin B12                              (11248) ⇢ 381 variants
  salicylic acid                           (9525) ⇢ 357 variants

✅ Acetaminophen (RxCUI 161): 1369 brand variants unified
